<a href="https://colab.research.google.com/github/Umama123/Machine-Learning-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# ==========================================
# SECTION 1: METHOD CHOICE & WHY
# ==========================================
"""
Method Choice: Random Forest Classifier
Why this method?
1. Non-linear patterns: The relationship between search position, impressions, and engagement metrics (GA4) is non-linear.
2. Robustness to Outliers: Search data has severe skewness; tree-based ensemble models handle extreme impression spikes gracefully.
3. Feature Importance: Random Forest allows straightforward feature importance and permutation evaluation required for audit.
"""
print("Section 1: Method choice documented (Random Forest Classifier chosen).")

Section 1: Method choice documented (Random Forest Classifier chosen).


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split Design Rationale: Why This Split is Honest**

1. **Class Imbalance Preservation** (stratify=y):High-opportunity CTR pages (is_ctr_opportunity == 1) are relatively rare compared to general search performance rows. By using a stratified split, we guarantee that both the training set (80%) and test set (20%) maintain the exact same class distribution, preventing biased evaluation.

2.  Fair Apples-to-Apples Comparison with Baseline:
To evaluate whether our Machine Learning model genuinely outperforms our Week 4 heuristic baseline, we evaluate both models on the exact same unseen $20\%$ test split using the same random seed (random_state=42).

3. No Data Leakage:Target labels and downstream metrics are strictly computed on $X_{train}$ during model training. The $X_{test}$ set remains completely unseen until final evaluation in Section 3.

In [4]:
# ==========================================
# SECTION 2: DATA SETUP & TRAIN-TEST SPLIT
# ==========================================
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

os.makedirs("work/outputs", exist_ok=True)

# 1. Fetch Hugging Face Token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Setup DuckDB & Create HTTP Secret with HF Bearer Token
conn = duckdb.connect()
conn.sql("INSTALL httpfs; LOAD httpfs;")
conn.sql(f"CREATE SECRET hf_secret (TYPE HTTP, BEARER_TOKEN '{hf_token}');")

# 3. Fetch Dataset
fact_path = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'"

df = conn.sql(f"""
    SELECT
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        COALESCE(ga4_pageviews, 0) as ga4_pageviews,
        COALESCE(ga4_sessions, 0) as ga4_sessions,
        COALESCE(ga4_engaged_sessions, 0) as ga4_engaged_sessions,
        COALESCE(sessions_organic, 0) as sessions_organic,
        ROUND(gsc_clicks::DOUBLE / NULLIF(gsc_impressions, 0), 4) as ctr,
        CASE
            WHEN gsc_avg_position <= 10.0 AND (gsc_clicks::DOUBLE / NULLIF(gsc_impressions, 0)) < 0.03 THEN 1
            ELSE 0
        END as is_ctr_opportunity
    FROM {fact_path}
    WHERE month = '2026-03'
      AND gsc_impressions > 0
      AND gsc_avg_position IS NOT NULL
""").df()

# 4. Features & Target Setup
features = ['gsc_impressions', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'sessions_organic']
X = df[features].fillna(0)
y = df['is_ctr_opportunity']

# 5. Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Data Loaded Successfully! Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data Loaded Successfully! Train Shape: (2888848, 5), Test Shape: (722213, 5)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
# ==========================================
# SECTION 3: MULTI-MODEL COMPARISON VS BASELINE
# ==========================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

print("Training multiple models on 2.88M rows...")

# 1. Logistic Regression (Fast Linear Model)
print("1/3 Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=500, random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# 2. Random Forest (Ensemble Tree Model)
print("2/3 Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# 3. Gradient Boosting (XGBoost Equivalent - Fast HistGradientBoosting)
print("3/3 Training Gradient Boosting...")
gb_model = HistGradientBoostingClassifier(max_iter=100, max_depth=10, random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

# 4. Baseline Rule (Week 4)
y_pred_baseline = ((X_test['gsc_avg_position'] <= 10.0) & (X_test['gsc_impressions'] >= 100)).astype(int)

# 5. Build Complete Comparison Table
metrics_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Week 4 Baseline': [
        accuracy_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_baseline, zero_division=0),
        recall_score(y_test, y_pred_baseline, zero_division=0),
        f1_score(y_test, y_pred_baseline, zero_division=0)
    ],
    'Logistic Regression': [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr, zero_division=0),
        recall_score(y_test, y_pred_lr, zero_division=0),
        f1_score(y_test, y_pred_lr, zero_division=0)
    ],
    'Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0)
    ],
    'Gradient Boosting': [
        accuracy_score(y_test, y_pred_gb),
        precision_score(y_test, y_pred_gb, zero_division=0),
        recall_score(y_test, y_pred_gb, zero_division=0),
        f1_score(y_test, y_pred_gb, zero_division=0)
    ]
}

comparison_df = pd.DataFrame(metrics_data)

print("\n================ FINAL MODEL COMPARISON TABLE ================")
print(comparison_df.round(4).to_string(index=False))

Training multiple models on 2.88M rows...
1/3 Training Logistic Regression...
2/3 Training Random Forest...
3/3 Training Gradient Boosting...

================ FINAL MODEL COMPARISON TABLE ================
   Metric  Week 4 Baseline  Logistic Regression  Random Forest  Gradient Boosting
 Accuracy           0.5300               0.9822         0.9894             0.9894
Precision           0.9957               0.9720         0.9848             0.9850
   Recall           0.2063               0.9988         0.9975             0.9973
 F1-Score           0.3418               0.9852         0.9911             0.9911


In [5]:
# ==========================================
# SECTION 3: TRAIN & COMPARE VS BASELINE
# ==========================================
print("Training Random Forest Classifier on 2.88M rows...")

# 1. Initialize & Fit Random Forest Model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Predict on unseen test set
y_pred_rf = rf_model.predict(X_test)

# 2. Evaluate Week 4 Heuristic Rule Baseline on identical X_test
# Week 4 Rule: High impression pages in top 10 positions
y_pred_baseline = ((X_test['gsc_avg_position'] <= 10.0) & (X_test['gsc_impressions'] >= 100)).astype(int)

# 3. Build Comparison Table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Week 4 Baseline Rule': [
        accuracy_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_baseline, zero_division=0),
        recall_score(y_test, y_pred_baseline, zero_division=0),
        f1_score(y_test, y_pred_baseline, zero_division=0)
    ],
    'ML Random Forest Model': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0)
    ]
})

print("\n--- MODEL VS BASELINE COMPARISON TABLE ---")
print(comparison_df.round(4).to_string(index=False))

Training Random Forest Classifier on 2.88M rows...

--- MODEL VS BASELINE COMPARISON TABLE ---
   Metric  Week 4 Baseline Rule  ML Random Forest Model
 Accuracy                0.5300                  0.9894
Precision                0.9957                  0.9848
   Recall                0.2063                  0.9975
 F1-Score                0.3418                  0.9911


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# ==========================================
# SECTION 4: FEATURE IMPORTANCE & ERROR ANALYSIS
# ==========================================
# 1. Feature Importances from Top Model (Random Forest)
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("--- FEATURE IMPORTANCE RANKING ---")
print(importance_df.to_string(index=False))

# 2. Error Inspection
errors_mask = (y_test != y_pred_rf)
df_errors = X_test[errors_mask].copy()
df_errors['True_Target'] = y_test[errors_mask]
df_errors['Predicted'] = y_pred_rf[errors_mask]

print(f"\nTotal Errors: {len(df_errors):,} out of {len(X_test):,} test samples ({len(df_errors)/len(X_test)*100:.2f}%)")
print("\nSample Error Cases:")
print(df_errors.head(5))

--- FEATURE IMPORTANCE RANKING ---
         Feature  Importance
gsc_avg_position    0.975512
 gsc_impressions    0.013222
sessions_organic    0.004991
   ga4_pageviews    0.004041
    ga4_sessions    0.002234

Total Errors: 7,653 out of 722,213 test samples (1.06%)

Sample Error Cases:
         gsc_impressions  gsc_avg_position  ga4_pageviews  ga4_sessions  \
3472143                9          4.777778              0             0   
2423217               24          0.583333              0             0   
1986789               25          4.880000              2             1   
1672447              192          3.286458              9             8   
485519                20          5.050000              0             0   

         sessions_organic  True_Target  Predicted  
3472143                 0            0          1  
2423217                 0            0          1  
1986789                 3            1          0  
1672447                 5            0          1  
48

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.